# Lab 5.4 · Linear Independence and Numerical Rank

Use the singular values to tell exact dependence from near dependence at a stated resolution.

Wooseok Ha, MAS110 Lecture 5, printed slides 26, 29–30, 40–42, and 60. The numerical tolerance experiment is a coding extension.

The lecture counts positive singular values to find the exact rank. With floating-point inputs, the count of useful directions depends on how small a scale can be distinguished from noise or roundoff.

Run the cells in order with NumPy. The code and notation follow the web lab.

## Notation

- $r=\operatorname{rank}(A)$ · Exact rank counts the positive singular values. The columns of an $n\times d$ matrix are independent exactly when $r=d$.

- $\tau$ · A numerical cutoff. Count $\sigma_i>\tau$ to obtain an effective rank. This example sets $\tau=\mathrm{rtol}\,\sigma_1$.

- $A^{\top}A$ · The feature-space Gram matrix. Its eigenvalues are $\sigma_i^2$, with additional zeros if needed to fill the $d$-dimensional space.

## 1. Compute singular values directly

The small entry is enough to give a second nonzero stretch.

**Predict first.** How far apart are the two scales?

In [ ]:
import numpy as np
eps = 1e-8
A = np.array([[1., 1.], [0., eps], [0., 0.]])
s = np.linalg.svd(A, compute_uv=False)
print("Direct singular values =", s)

The second stretch is around eight orders of magnitude smaller than the first.

## 2. Compare two relative cutoffs

The loose cutoff treats the nearly canceled direction as zero. The tighter cutoff retains it.

**Predict first.** Does multiplying every entry by a million change either count?

In [ ]:
rtols = np.array([1e-6, 1e-10])
ranks = [np.count_nonzero(s > t * s[0]) for t in rtols]
s_scaled = np.linalg.svd(1e6 * A, compute_uv=False)
scaled_ranks = [np.count_nonzero(s_scaled > t * s_scaled[0]) for t in rtols]
print("Cutoffs =", rtols, "\nRanks =", ranks, "\nRescaled ranks =", scaled_ranks)

Each row compares the original and rescaled matrices. The first row uses the looser cutoff and the second uses the tighter cutoff.

## 3. Compare the Gram matrix

The squared perturbation rounds away when added to the bottom-right entry.

**Predict first.** Which matrix now appears rank one in floating point?

In [ ]:
G = A.T @ A
gram_eigenvalues = np.linalg.eigvalsh(G)
default_rank = np.linalg.matrix_rank(A)
print("Rounded Gram matrix =", G, "\nGram eigenvalues =", gram_eigenvalues, "\nDefault rank =", default_rank)

The rounded Gram matrix has eigenvalues zero and two. The default rank of the original matrix is still two.

## When a direction disappears from the computation

### A small stretch is still a stretch

The second column is the first column plus $\varepsilon$ in its second entry. For $\varepsilon\ne 0$ this creates a new direction. A unit vector close to $(1,-1)^{\top}/\sqrt{2}$ nearly cancels the two columns, so the smallest singular value is tiny.

$$\left\lVert A(1,-1)^{\top}/\sqrt{2}\right\rVert_2=|\varepsilon|/\sqrt{2}$$

### Use the scale of the input when setting a cutoff

Multiplying $A$ by a nonzero scalar multiplies all singular values by the magnitude of that scalar. A relative cutoff scales with them. NumPy’s default matrix-rank threshold for this example is the largest singular value times the larger dimension times machine epsilon. A data-specific noise level can call for a different cutoff.

$$\tau_{\mathrm{default}}=\sigma_1\max(n,d)\,\varepsilon_{\mathrm{mach}}$$

### The Gram identity is exact algebra

In exact arithmetic $A^{\top}Av_i=\sigma_i^2v_i$. In this experiment, forming the Gram matrix adds $10^{-16}$ to $1$, which rounds back to $1$ in double precision. The direct SVD still sees a nonzero small direction. The eigenvalue identity explains the connection, and computing the SVD directly avoids forming this rounded Gram matrix.

$$\lambda_i(A^{\top}A)=\sigma_i^2$$

## Verify the example

Run these checks on the example before changing the input.

In [ ]:
assert np.allclose(s, [np.sqrt(2), eps / np.sqrt(2)], rtol=1e-6, atol=0)
assert ranks == scaled_ranks == [1, 2]
assert np.allclose(s_scaled, 1e6 * s, atol=0)
assert np.array_equal(G, np.ones((2, 2)))
assert np.allclose(gram_eigenvalues, [0, 2], atol=1e-14)
assert default_rank == 2
A_dependent = A.copy()
A_dependent[1, 1] = 0
assert np.linalg.matrix_rank(A_dependent) == 1
assert np.allclose(A_dependent @ [1, -1], 0)

## Check your understanding

What if $\varepsilon=0$ exactly?

The columns coincide, $(1,-1)^{\top}$ is in the null space, and the exact rank is one.

Does changing the cutoff change the matrix’s exact rank?

The cutoff changes the effective rank of the computation. At the stated nonzero $\varepsilon$ the exact rank is two under either cutoff.

## References

- [NumPy numerical rank](https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_rank.html)
- [Original Chapter 5.4 notebook](https://github.com/kyunghyuncho/Foundations_of_LADS/blob/main/Ch5-4%20Test%20of%20Linear%20Independence.ipynb)